# World Cup & International Football — Exploratory Data Analysis

International men's football results from **1872 to 2026** (~49k matches), enriched
with goal-level events, Elo ratings and per-edition World Cup data.

**Goal:** understand the data, surface its quirks, and extract the historical
patterns that feed a *2026 World Cup forecasting* model (see [`experiments/`](../experiments)).

The stack is **Polars** end to end: loading/cleaning lives in
[`src/worldcup/data.py`](../src/worldcup/data.py), and every figure / LaTeX table is
produced through the NeurIPS-styled [`worldcup.viz`](../src/worldcup/viz) layer.

In [1]:
import sys; sys.path.insert(0, "../src")
import polars as pl
from worldcup import data, viz
from worldcup.viz import plots
viz.set_style()

## 1. Match results
One row per international fixture.

In [2]:
results = data.load_results()
played = data.load_results(played_only=True)
print(f"{results.height:,} fixtures | {results['date'].min()} -> {results['date'].max()}")
print(f"played: {played.height:,} | future/unplayed: {results.height - played.height}")
results.head()

49,477 fixtures | 1872-11-30 -> 2026-06-27
played: 49,413 | future/unplayed: 64


date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
date,str,str,i64,i64,str,str,str,bool
1872-11-30,"""Scotland""","""England""",0,0,"""Friendly""","""Glasgow""","""Scotland""",false
1873-03-08,"""England""","""Scotland""",4,2,"""Friendly""","""London""","""England""",false
1874-03-07,"""Scotland""","""England""",2,1,"""Friendly""","""Glasgow""","""Scotland""",false
1875-03-06,"""England""","""Scotland""",2,2,"""Friendly""","""London""","""England""",false
1876-03-04,"""Scotland""","""England""",3,0,"""Friendly""","""Glasgow""","""Scotland""",false


In [3]:
display(results.group_by("tournament").len().sort("len", descending=True).head(8))
g = played["total_goals"]
print(f"goals/match: mean {g.mean():.2f} | median {g.median():.0f} | max {g.max()}")

tournament,len
str,u32
"""Friendly""",18388
"""FIFA World Cup qualification""",8771
"""UEFA Euro qualification""",2824
"""African Cup of Nations qualifi…",2327
"""FIFA World Cup""",1036
"""Copa América""",869
"""African Cup of Nations""",845
"""AFC Asian Cup qualification""",829


goals/match: mean 2.94 | median 3 | max 31


In [4]:
viz.save_fig(plots.matches_per_year(played), "01_matches_per_year")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/01_matches_per_year.pdf')

**Goals per match** collapsed from 5–9 in the 1870s–1900s and has been flat at
~2.7 since the 1960s — football professionalized and defenses tightened.

In [5]:
viz.save_fig(plots.goals_per_match_trend(played), "02_goals_per_match")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/02_goals_per_match.pdf')

In [6]:
viz.save_fig(plots.goals_distribution(played), "03_goals_distribution")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/03_goals_distribution.pdf')

## 2. Home advantage
Restricting to **non-neutral** venues isolates the effect.

In [7]:
nz = played.filter(~pl.col("neutral"))
outcome = nz.select(
    home=(pl.col("home_score") > pl.col("away_score")).mean() * 100,
    draw=(pl.col("home_score") == pl.col("away_score")).mean() * 100,
    away=(pl.col("home_score") < pl.col("away_score")).mean() * 100)
print(outcome)

shape: (1, 3)
┌───────────┬───────────┬─────────┐
│ home      ┆ draw      ┆ away    │
│ ---       ┆ ---       ┆ ---     │
│ f64       ┆ f64       ┆ f64     │
╞═══════════╪═══════════╪═════════╡
│ 50.742779 ┆ 22.858322 ┆ 26.3989 │
└───────────┴───────────┴─────────┘


In [8]:
viz.save_fig(plots.home_advantage_by_decade(played), "04_home_advantage")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/04_home_advantage.pdf')

Home advantage (~51% home wins) is stable across 120 years; the share of
**draws** is what crept up over time.

## 3. Team performance
Per-team match log → historical points percentage. Exported as a NeurIPS table.

In [9]:
log = data.team_match_log(played)
best = (log.group_by("team").agg(
            matches=pl.len(), wins=pl.col("win").sum(),
            points_pct=((pl.col("win")*3 + pl.col("draw")).sum() / (pl.len()*3) * 100).round(1))
        .filter(pl.col("matches") >= 200).sort("points_pct", descending=True).head(10))
best

team,matches,wins,points_pct
str,u32,u32,f64
"""Brazil""",1060,672,70.2
"""Jersey""",235,153,67.9
"""Spain""",783,461,66.6
"""England""",1090,625,65.2
"""Germany""",1031,599,65.0
"""Iran""",612,349,64.9
"""Guernsey""",240,145,63.9
"""Argentina""",1069,592,63.4
"""Italy""",893,477,62.5


In [10]:
tex = viz.df_to_neurips_latex(
    best, label="tab:winrate",
    caption=("Historical points percentage of the top national teams "
             "(minimum 200 matches; higher is better). Brazil leads despite the largest load."),
    float_format="%.1f", bold_best="points_pct", lower_is_better=False)
print(viz.save_table(tex, "winrate"))

C:\Users\luisg\worldcup\reports\tables\winrate.tex


## 4. Goal events
Scorers, penalties and goal timing.

In [11]:
goals = data.load_goalscorers()
print(f"{goals.height:,} goals | penalties {goals['penalty'].mean()*100:.1f}% | "
      f"own goals {goals['own_goal'].mean()*100:.1f}%")
display(goals.filter(~pl.col("own_goal")).group_by("scorer").len()
            .sort("len", descending=True).head(10))

47,620 goals | penalties 6.8% | own goals 1.9%


scorer,len
str,u32
"""Cristiano Ronaldo""",121
"""Robert Lewandowski""",69
"""Harry Kane""",69
"""Romelu Lukaku""",64
"""Lionel Messi""",63
"""Edin Džeko""",58
"""Aleksandar Mitrović""",52
"""Luis Suárez""",51
"""Ali Daei""",49


In [12]:
viz.save_fig(plots.goal_minute_distribution(goals), "05_goal_minute")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/05_goal_minute.pdf')

## 5. Elo ratings
`load_elo` fixes the mixed ISO/US date encoding; `latest_elo` drops dissolved
nations (e.g. *West Germany*) from the current ranking.

In [13]:
latest = data.latest_elo(exclude_dissolved=True)
viz.save_fig(plots.top_elo(latest), "07_top_elo")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/07_top_elo.pdf')

In [14]:
elo = data.load_elo()
viz.save_fig(plots.elo_history(elo, ["Brazil","Spain","Argentina","England"]), "08_elo_history")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/08_elo_history.pdf')

## 6. World Cup editions
Goals per match per tournament, 1930–2026.

In [15]:
editions = data.load_worldcup_editions()
viz.save_fig(plots.worldcup_goals_per_match(editions), "06_worldcup_goals")

WindowsPath('C:/Users/luisg/worldcup/reports/figures/06_worldcup_goals.pdf')

In [16]:
tex = viz.df_to_neurips_latex(
    editions, label="tab:wcgoals",
    caption=("Goals per match for every World Cup edition (1930–2026). "
             "Scoring peaked in 1954 and has hovered near 2.6 since the 1990s."),
    float_format="%.2f")
print(viz.save_table(tex, "worldcup_editions"))
editions

C:\Users\luisg\worldcup\reports\tables\worldcup_editions.tex


year,matches,goals,goals_per_match
i64,i64,i64,f64
1930,18,70,3.888889
1934,17,66,3.882353
1938,18,75,4.166667
1950,22,88,4.0
1954,26,136,5.230769
…,…,…,…
2010,64,143,2.234375
2014,64,163,2.546875
2018,64,166,2.59375


## Key takeaways
1. **Scoring** fell sharply early on, stable at ~2.7 goals/match since the 1960s.
2. **Home advantage** is durable (~51% home wins); draws rose over time.
3. **Brazil** leads historical points% among heavily-tested sides; **Spain** tops current Elo.
4. World Cup scoring peaked in **1954 (5.2)**, bottomed in **1990 (2.1)**, ~2.6 today.

### Data-quality notes (handled in `src/worldcup/data.py`)
- Elo dates come in two formats — single-format parsing nulls ~99% of rows.
- `results` includes future fixtures (NA scores) — filtered via `played_only`.
- Dissolved nations coexist with current ones — see `latest_elo(exclude_dissolved=True)`.

**Next:** the clean Parquet bases (`worldcup.clean`) feed an Elo–Poisson hybrid to
forecast the 2026 World Cup → [`experiments/`](../experiments).